## Ch8-01 — Satisfaction evaluation

This notebook introduces `verify_satisfaction()` to evaluate `assert satisfy` declarations; after running it you can confirm which design variants satisfy the TimelyToast requirement and which do not.


Chapter 3 declared `assert satisfy timely by nominal` and `assert satisfy timely by slow`. This notebook calls `verify_satisfaction()` to evaluate both declarations computationally, producing `Verdict` objects that report whether each candidate holds. See [Ch3-01 MoE definition](../ch03-measures/01-moe-definition.ipynb) for the requirement and satisfy declarations.


In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    requirement def HeatingReq {
        subject heater : Heater;
        require constraint { heater.power >= 600.0 }
    }
    requirement heating : HeatingReq;
    part efficient : Heater;
    part weak : Heater { attribute :>> power = 400.0; }
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement {
        attribute resistance : Real default = 12.0;
    }
    part def PowerWire :> HeatingElement {
        attribute gauge : Real default = 14.0;
    }
    part def HeatingAssembly :> HeatingSystem {
        part coil : ResistanceCoil;
        part wire : PowerWire;
    }
    part heatingEvidence {
        assert satisfy heating by efficient;
        assert satisfy heating by weak;
    }
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    state Cycle {
        entry; then idle;
        state idle;
        state heating;
        state ready;
        state cancelled;
        transition first idle accept Start then heating;
        transition first heating accept Finish then ready;
        transition first heating accept Cancel then cancelled;
    }

    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")


In [ ]:
# A require constraint referencing an undefined attribute fails.
bad_source = """
package P {
    private import ScalarValues::*;
    part def Thing { attribute x : Real default = 5.0; }
    requirement def Check {
        subject t : Thing;
        require constraint { t.undeclared <= 10.0 }
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for undeclared attribute in constraint"
# Expected: diagnostic for 'undeclared' as an unresolved attribute reference
print(f"Negative control ok: bad.ok={bad.ok}")


In [ ]:
# Evaluate all assert-satisfy declarations in the model
verdicts = model.verify_satisfaction()
print(f"Verdicts returned: {len(verdicts)}")
for v in verdicts:
    status = "PASS" if v.holds else "FAIL"
    print(f"  [{status}] {v.element}")

# Check the TimelyToast verdicts specifically
timely_verdicts = [v for v in verdicts if "timely" in (v.element or "").lower()]
nominal_verdict = next((v for v in timely_verdicts if "nominal" in (v.element or "")), None)
slow_verdict    = next((v for v in timely_verdicts if "slow"    in (v.element or "")), None)

assert nominal_verdict is not None, "Nominal verdict not found"
assert slow_verdict    is not None, "Slow verdict not found"
assert nominal_verdict.holds is True,  f"Expected nominal to hold: {nominal_verdict}"
assert slow_verdict.holds    is False, f"Expected slow to fail: {slow_verdict}"

print(f"\nnominal holds={nominal_verdict.holds}  (cycleTime=120 ≤ 180)")
print(f"slow    holds={slow_verdict.holds}   (cycleTime=200 > 180)")
conn.close()


The `assert satisfy timely by nominal` and `assert satisfy timely by slow` declarations (A-F) are evaluated by `verify_satisfaction()` in OpenSysML (O-S); the Verdict objects show `nominal` holds=True and `slow` holds=False (E).


Try the chapter exercise in `exercises/ch08/exercise.ipynb`: add a `fast : Toaster` variant with `cycleTime = 90.0` and confirm via `verify_satisfaction()` that it also satisfies the TimelyToast requirement.
